In [ ]:
import sys
import pandas as pd, numpy as np
import cohort_table as ct

panel  = pd.read_csv(ct.PANEL)
lookup = pd.read_csv(ct.LOOKUP, usecols=[ct.ID, "Year", "service_option"])
df     = ct.build_outcome(panel, lookup)
len(df), df.returned.mean()

(130417, np.float64(0.5939793125129392))

In [10]:
# 1. drop CJO
bk = df[df.borough.eq("Brooklyn")]
CJO = "Council of Jewish Organization"
print(f"Brooklyn all:      {bk.returned.mean():.3f}  n={len(bk):,}")
print(f"Brooklyn minus CJO:{bk[~bk.provider.eq(CJO)].returned.mean():.3f}  "
      f"n={(~bk.provider.eq(CJO)).sum():,}")
print(f"Pooled:            {df.returned.mean():.3f}")

print('\n')

# 2. do providers span boroughs?
ct_pb = pd.crosstab(df.provider, df.borough)
print(ct_pb.div(ct_pb.sum(1), axis=0).round(2).head(15))
print("providers >90% in one borough:",
      (ct_pb.div(ct_pb.sum(1), axis=0).max(axis=1) > .9).mean().round(2))

print('\n')

# 3. direct standardization to the pooled provider mix
d = df.assign(provider=df.provider.fillna("(missing)"))

w    = d.groupby("provider").size() / len(d)          # sums to 1
rate = d.groupby(["borough", "provider"]).returned.mean().unstack()
present = rate.notna()

num = (rate.fillna(0) * w).sum(axis=1)
den = (present * w).sum(axis=1)
adj = num / den

out = pd.DataFrame({"crude": d.groupby("borough").returned.mean(),
                    "adjusted": adj,
                    "coverage": den,
                    "n": d.groupby("borough").size()})
out["shift"] = (out.adjusted - out.crude).round(3)

# sanity: size-weighted adjusted should land near the pooled rate
print(out.round(3))
print("check:", np.average(out.adjusted, weights=out.n).round(3), "vs", d.returned.mean().round(3))

Brooklyn all:      0.628  n=57,895
Brooklyn minus CJO:0.589  n=40,572
Pooled:            0.594


borough                         Bronx  Brooklyn  Manhattan  Queens  \
provider                                                             
Aspira of New York               0.95      0.01       0.03    0.00   
BCS                              0.00      0.95       0.01    0.02   
Boys & Girls Club Queens         0.01      0.18       0.00    0.81   
Bridge Street                    0.01      0.93       0.01    0.04   
BronxWorks, Inc                  0.96      0.00       0.03    0.00   
Brooklyn Neighborhood Improvem   0.01      0.94       0.01    0.03   
CAMBA                            0.01      0.96       0.01    0.01   
CCCS                             0.62      0.06       0.30    0.02   
Catholic Charities Neighborhoo   0.04      0.61       0.02    0.32   
Center for Family Life           0.00      0.93       0.01    0.01   
Children's Arts & Science Work   0.88      0.03       0.08    0

<positron-console-cell-10>:13: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
<positron-console-cell-10>:15: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.


Brooklyn's higher rate is partly provider composition — one large provider operating mostly in Brooklyn returns at 72% — but standardizing every borough to the same provider mix only closes about a third of the gap, so most of it is something else.